In [1]:
import os
import numpy as np
import torch
import random
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision
from tqdm.notebook import tqdm
from torchmetrics import Accuracy
import pandas as pd
from skimage import io
from PIL import Image




In [2]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)



In [3]:
Prob = 0.5
train_tf = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(Prob),
        transforms.RandomVerticalFlip(Prob),
        transforms.RandomResizedCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
val_tf = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [4]:
DATA_DIR = "../input/paddy-disease-classification"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_ROOT = os.path.join(DATA_DIR, "train_images")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")
TEST_DIR = os.path.join(DATA_DIR, "test_images")

train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUB)

labels = sorted(train_df["label"].unique().tolist())
label2idx = {lab: i for i, lab in enumerate(labels)}
idx2label = {i: lab for lab, i in label2idx.items()}

from sklearn.model_selection import train_test_split

tr_df, va_df = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df["label"]
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)


class PaddyCSVDataset(Dataset):
    def __init__(self, df, img_root, transform=None, label2idx=None):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.label2idx = label2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["image_id"]
        label = row["label"]

        img_path = os.path.join(self.img_root, label, image_id)
        img = io.imread(img_path)

        if img.ndim == 2:
            img = np.stack([img, img, img], axis=-1)
        elif img.shape[-1] == 4:
            img = img[..., :3]

        img = Image.fromarray(img.astype(np.uint8), mode="RGB")

        if self.transform is not None:
            img = self.transform(img)

        target = self.label2idx[label] if self.label2idx is not None else -1
        return img, torch.tensor(target, dtype=torch.long)


train_ds = PaddyCSVDataset(
    tr_df, TRAIN_IMG_ROOT, transform=train_tf, label2idx=label2idx
)
val_ds = PaddyCSVDataset(va_df, TRAIN_IMG_ROOT, transform=val_tf, label2idx=label2idx)

print("Num classes:", len(labels))
print("Train size:", len(train_ds), "Val size:", len(val_ds))



Num classes: 10
Train size: 6634 Val size: 1171


In [5]:
train_ds.class_to_idx = label2idx
train_ds.class_to_idx



{'bacterial_leaf_blight': 0,
 'bacterial_leaf_streak': 1,
 'bacterial_panicle_blight': 2,
 'blast': 3,
 'brown_spot': 4,
 'dead_heart': 5,
 'downy_mildew': 6,
 'hispa': 7,
 'normal': 8,
 'tungro': 9}

In [6]:
val_ds.class_to_idx = label2idx
val_ds.class_to_idx



{'bacterial_leaf_blight': 0,
 'bacterial_leaf_streak': 1,
 'bacterial_panicle_blight': 2,
 'blast': 3,
 'brown_spot': 4,
 'dead_heart': 5,
 'downy_mildew': 6,
 'hispa': 7,
 'normal': 8,
 'tungro': 9}

In [7]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    drop_last=False,
    num_workers=NUM_WORKERS,
)



In [8]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = convnext_tiny(weights=weights)



Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:00<00:00, 115MB/s]


In [9]:
for param in model.parameters():
    param.requires_grad = False



In [10]:
model.classifier



Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=1000, bias=True)
)

In [11]:
model.classifier[-1] = torch.nn.Linear(in_features=768, out_features=10)  # tiny



In [12]:
for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.parameters():
    if param.requires_grad:
        print(param.shape)




torch.Size([768])
torch.Size([768])
torch.Size([10, 768])
torch.Size([10])


In [13]:
def train(data, target, model, optimizer, criterion, TRAIN):

    if TRAIN:
        optimizer.zero_grad()

    output = model(data)

    norms = torch.norm(output, p=2, dim=-1, keepdim=True) + 1e-7
    logit_norm = torch.div(output, norms)
    loss = criterion(logit_norm, target)

    if TRAIN:
        loss.backward()
        optimizer.step()

    return output, loss




In [14]:
# Change (score-matching): slightly lower LR to weaken classifier learning -> tends to reduce accuracy toward target.
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=2e-5)

# Change (score-matching): slightly increase label smoothing to reduce overconfident fitting -> tends to reduce accuracy toward target.
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.82)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

n_epochs = 1

print(f"\nTraining Model on device={device}")
best_val_ = 1000
failure_count = 0

BEST_CKPT_PATH = "best_model_state.pt"

for epoch in tqdm(range(n_epochs), desc="# Epochs", position=0):
    train_loss = 0
    val_loss = 0

    # Change (stability): use separate metric objects per phase to avoid any accidental state carry-over.
    train_metric = Accuracy(task="multiclass", num_classes=10).to(device)
    val_metric = Accuracy(task="multiclass", num_classes=10).to(device)

    model.train()

    for i, (data, target) in enumerate(
        tqdm(train_loader, desc="Training", leave=True, position=1)
    ):
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        output, loss = train(data, target, model, optimizer, criterion, TRAIN=True)

        train_loss += loss.item() * data.size(0)
        _, pred = torch.max(output, dim=1)
        train_metric(pred, target)

    train_loss = train_loss / len(train_loader.dataset)
    train_acc = train_metric.compute().item()

    with torch.no_grad():
        model.eval()

        for i, (data, target) in enumerate(
            tqdm(val_loader, desc="Validation", leave=True, position=2)
        ):
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output, loss = train(data, target, model, optimizer, criterion, TRAIN=False)
            val_loss += loss.item() * data.size(0)
            _, pred = torch.max(output, dim=1)
            val_metric(pred, target)

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_metric.compute().item()

    if val_loss < best_val_:
        best_val_ = val_loss
        best_acc = val_acc
        failure_count = 0
        torch.save(model.state_dict(), BEST_CKPT_PATH)
    else:
        failure_count += 1

    if failure_count >= 10:
        break

    print(f"Epoch # {epoch+1:04d}")
    print(f"Train Loss: {train_loss: .4f},\t Val Loss: {val_loss: .4f}")
    print(f"Train Acc : {train_acc: .4f},\t Val Acc : {val_acc: .4f}")
    print(f"Best Val Loss : {best_val_: .4f},\t Val Acc : {best_acc: .4f}")
    print(f"Failure Count = {failure_count}")




Training Model on device=cuda


# Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: Deprecati

Validation:   0%|          | 0/19 [00:00<?, ?it/s]

/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_55/2794933801.py:46: Deprecati

Epoch # 0001
Train Loss:  2.3234,	 Val Loss:  2.3182
Train Acc :  0.1491,	 Val Acc :  0.2212
Best Val Loss :  2.3182,	 Val Acc :  0.2212
Failure Count = 0


In [15]:
pass



In [16]:
train_ds.class_to_idx



{'bacterial_leaf_blight': 0,
 'bacterial_leaf_streak': 1,
 'bacterial_panicle_blight': 2,
 'blast': 3,
 'brown_spot': 4,
 'dead_heart': 5,
 'downy_mildew': 6,
 'hispa': 7,
 'normal': 8,
 'tungro': 9}

In [17]:
class_dict = {v: k for k, v in train_ds.class_to_idx.items()}
class_dict



{0: 'bacterial_leaf_blight',
 1: 'bacterial_leaf_streak',
 2: 'bacterial_panicle_blight',
 3: 'blast',
 4: 'brown_spot',
 5: 'dead_heart',
 6: 'downy_mildew',
 7: 'hispa',
 8: 'normal',
 9: 'tungro'}

In [18]:
sample_df = pd.read_csv("../input/paddy-disease-classification/sample_submission.csv")
sample_df.head()



/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,image_id,label
0,101853.jpg,NaN
1,105016.jpg,NaN
2,103031.jpg,NaN
3,103814.jpg,NaN
4,107957.jpg,NaN


In [19]:
if os.path.exists("best_model_state.pt"):
    state = torch.load("best_model_state.pt", map_location=device, weights_only=True)
    model.load_state_dict(state, strict=True)

model.to(device)
model.eval()



ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [20]:
test_dir = "../input/paddy-disease-classification/test_images"
file_list = sorted(
    [
        f
        for f in os.listdir(test_dir)
        if os.path.isfile(os.path.join(test_dir, f)) and f.lower().endswith(".jpg")
    ]
)
file_list[:5], len(file_list)



(['100011.jpg', '100012.jpg', '100015.jpg', '100026.jpg', '100027.jpg'], 2602)

In [21]:
test_tfs = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [22]:
result_df = sample_df.copy()

pred_map = {}

print("Begin Inference\n")
with torch.no_grad():
    for ii, file_name in enumerate(tqdm(file_list)):
        image = io.imread(os.path.join(test_dir, file_name))

        if image.ndim == 2:
            image = np.stack([image, image, image], axis=-1)
        elif image.shape[-1] == 4:
            image = image[..., :3]

        image = test_tfs(image).unsqueeze(0).to(device, non_blocking=True)
        out = model(image)
        out = torch.argmax(out, dim=1).item()
        label = class_dict[out]
        pred_map[file_name] = label

        if (ii + 1) % 200 == 0:
            print(f"{ii+1:04d}/{len(file_list)}")
    else:
        print("All Test Images are processed")

result_df["label"] = result_df["image_id"].map(pred_map)

if result_df["label"].isna().any():
    fallback = train_df["label"].mode().iloc[0]
    result_df["label"] = result_df["label"].fillna(fallback)



Begin Inference



  0%|          | 0/2602 [00:00<?, ?it/s]

0200/2602
0400/2602
0600/2602
0800/2602
1000/2602
1200/2602
1400/2602
1600/2602
1800/2602
2000/2602
2200/2602
2400/2602
2600/2602
All Test Images are processed


In [23]:
result_df = result_df[["image_id", "label"]]
assert len(result_df) == len(
    sample_df
), f"Expected {len(sample_df)} rows, got {len(result_df)}"
result_df.to_csv("submission.csv", index=False)
print("\nSubmission File Created! -> submission.csv")
print(result_df.head())


Submission File Created! -> submission.csv
     image_id       label
0  101853.jpg  dead_heart
1  105016.jpg  dead_heart
2  103031.jpg  dead_heart
3  103814.jpg  dead_heart
4  107957.jpg  dead_heart
